In [1]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

In [2]:
from dotenv import load_dotenv, find_dotenv
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
import os
from llama_index.embeddings.openai import OpenAIEmbedding

# Load environment variables
load_dotenv(find_dotenv())  
embed_model = OpenAIEmbedding(model_name = os.environ.get("EMBED_MODEL"))

if embed_model is None:
    raise ValueError("EMBED_MODEL environment variable is not set!")

# Load documents
documents = SimpleDirectoryReader(input_files=["../data/Pmg_lds.md"]).load_data()



In [3]:
# merge into a single large document rather than the one document per page

from llama_index.core import Document

document = Document(text="\n\n".join([doc.text for doc in documents]))

In [4]:
from llama_index.core.node_parser import HierarchicalNodeParser

node_parser = HierarchicalNodeParser.from_defaults(chunk_sizes=[4096, 2048, 512])

In [5]:
nodes = node_parser.get_nodes_from_documents([document])

In [6]:
from llama_index.core.node_parser import get_leaf_nodes

leaf_nodes = get_leaf_nodes(nodes)


In [8]:
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.core import load_index_from_storage

if not os.path.exists("../VectorStore"):
    os.makedirs("../VectorStore")
    storage_context = StorageContext.from_defaults()
    storage_context.docstore.add_documents(nodes)

    automerging_index = VectorStoreIndex(
        leaf_nodes, storage_context=storage_context)

    automerging_index.storage_context.persist(persist_dir="../VectorStore")

else:
    automerging_index = load_index_from_storage(StorageContext.from_defaults(persist_dir="../VectorStore"))

In [9]:
from llama_index.postprocessor.cohere_rerank import CohereRerank
load_dotenv(find_dotenv())
cohere_rerank = CohereRerank(
    api_key=os.environ["COHERE_API_KEY"], 
    top_n=2,
)

In [10]:
from llama_index.core.retrievers import AutoMergingRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

automerging_retriever = automerging_index.as_retriever(similarity_top_k=6)

retriever = AutoMergingRetriever(
    automerging_retriever,
    automerging_index.storage_context,
    verbose= True
)

auto_merging_engine = RetrieverQueryEngine.from_args(
    retriever, node_postprocessors=[cohere_rerank]
)

### Evaluation

In [11]:
from trulens_eval import Tru

tru = Tru()
tru.reset_database()


/tmp/ipykernel_3120/3827208637.py:1: DeprecationWarning: The `trulens_eval` module is deprecated. See https://trulens.org/docs/trulens/guides/trulens_eval_migration for instructions on migrating to `trulens.*` modules.
  from trulens_eval import Tru
/tmp/ipykernel_3120/3827208637.py:3: DeprecationWarning: Class `TruSession` has moved:
	New import: `from trulens.core.session import TruSession`
 See https://trulens.org/docs/trulens/guides/trulens_eval_migration for instructions on migrating to `trulens` modules.
  tru = Tru()


🦑 TruSession initialized with db url sqlite:///default.sqlite .
🛑 Secret keys may be written to the database. See the `database_redact_keys` option of `TruSession` to prevent this.


In [12]:
import numpy as np
from trulens.apps.llamaindex import TruLlama
from trulens.core import Feedback
from trulens.providers.openai import OpenAI

# Initialize provider class
provider = OpenAI(model_engine="gpt-4o-mini")

# select context to be used in feedback. the location of context is app specific.

context = TruLlama.select_context(auto_merging_engine)

# Define a groundedness feedback function
f_groundedness = (
    Feedback(
        provider.groundedness_measure_with_cot_reasons, name="Groundedness"
    )
    .on(context.collect())  # collect context chunks into a list
    .on_output()
)

# Question/answer relevance between overall question and answer.
f_answer_relevance = Feedback(
    provider.relevance_with_cot_reasons, name="Answer Relevance"
).on_input_output()
# Question/statement relevance between question and each context chunk.
f_context_relevance = (
    Feedback(
        provider.context_relevance_with_cot_reasons, name="Context Relevance"
    )
    .on_input()
    .on(context)
    .aggregate(np.mean)
)

✅ In Groundedness, input source will be set to __record__.app.query.rets.source_nodes[:].node.text.collect() .
✅ In Groundedness, input statement will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Answer Relevance, input prompt will be set to __record__.main_input or `Select.RecordInput` .
✅ In Answer Relevance, input response will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Context Relevance, input question will be set to __record__.main_input or `Select.RecordInput` .
✅ In Context Relevance, input context will be set to __record__.app.query.rets.source_nodes[:].node.text .


In [13]:
from trulens.apps.llamaindex.guardrails import WithFeedbackFilterNodes

# note: feedback function used for guardrail must only return a score, not also reasons
f_context_relevance_score = Feedback(provider.context_relevance)

filtered_query_engine = WithFeedbackFilterNodes(
    auto_merging_engine, feedback=f_context_relevance_score, threshold=0.5
)

In [14]:
tru_query_engine_recorder = TruLlama(
    auto_merging_engine,
    app_name="LlamaIndex_App",
    app_version="03_Advanced_Retriever",
    feedbacks=[f_groundedness, f_answer_relevance, f_context_relevance],
)

🦑 TruSession initialized with db url sqlite:///default.sqlite .
🛑 Secret keys may be written to the database. See the `database_redact_keys` option of `TruSession` to prevent this.


In [15]:
import pandas as pd

# Read the CSV file and drop the 'Unnamed: 0' column
df = pd.read_csv("../dataset_eval/20_dataset.csv").drop(columns=['Unnamed: 0'])

# Extract the first 20 questions into a list
questions = df['question'][:20].tolist()

# Print the list of questions
print(questions)


['What strategies can be used to make a message easy to understand when teaching?', 'What should teachers do with unfamiliar words to ensure their message is easy to understand?', 'What are some effective study techniques to enhance understanding and retention of material?', 'What is the purpose of using a study journal in your scripture study?', 'What is the importance of organizing and summarizing lesson plans for effective teaching?', 'What is the recommended method for highlighting key words when marking scriptures?', 'What is the significance of beginning study activities with a prayer?', 'What is the significance of the power of ordination in the context of missionary work?', 'What is the purpose of marking scriptures in relation to applying gospel teachings?', 'What is the significance of preaching the gospel according to President Lorenzo Snow?', 'What is the relationship between the Atonement and missionary work according to President Howard W. Hunter?', 'What are the blessing

In [16]:
# evaluate
for question in questions:
    with tru_query_engine_recorder as recording:
        response = auto_merging_engine.query(question)

> Merging 3 nodes into parent node.
> Parent node id: 63b7ba91-684f-4b9b-b13f-7004075fbda2.
> Parent node text: Use Study Resources (Page 37)
- Use the study aids in the LDS edition of the scriptures (Topical ...

> Merging 3 nodes into parent node.
> Parent node id: 4a11252f-bcc1-40a8-8ff2-ff1081645219.
> Parent node text: Marking Scriptures (Page 38)

Marking your scriptures can assist you in thinking deeply about a p...

> Merging 1 nodes into parent node.
> Parent node id: a0ea3ebd-aa2b-4849-8414-fe3420d454c7.
> Parent node text: Cultural Views of Scriptures (Page 195)
Most religions have sacred texts or books of scripture, b...

> Merging 3 nodes into parent node.
> Parent node id: 4a11252f-bcc1-40a8-8ff2-ff1081645219.
> Parent node text: Marking Scriptures (Page 38)

Marking your scriptures can assist you in thinking deeply about a p...



In [17]:
records, feedback = tru.get_records_and_feedback(app_ids=[])
# records.head()

In [18]:
import pandas as pd

pd.set_option("display.max_colwidth", None)
records[["input", "output"] + feedback]

,input,output,Context Relevance,Groundedness,Answer Relevance
0,"""What strategies can be used to make a message easy to understand when teaching?""","""Strategies that can be used to make a message easy to understand when teaching include defining unfamiliar words simply, using language that is accessible to the audience, studying key definitions and terms beforehand, becoming familiar with provided definitions, and utilizing resources like True to the Faith and the Bible Dictionary for additional explanations.""",0.666667,1.000000,1.000000
1,"""What should teachers do with unfamiliar words to ensure their message is easy to understand?""","""Teachers should learn how to define unfamiliar words simply and make the message easy to understand for those they are teaching.""",1.000000,1.000000,0.666667
2,"""What are some effective study techniques to enhance understanding and retention of material?""","""Effective study techniques to enhance understanding and retention of material include exercising, showering, and praying before studying, studying at a desk or table, organizing study materials, studying by topic, asking questions about the central message and personal application, visualizing the content being studied, and engaging in additional learning activities such as personal study, companion study, district meetings, and zone conferences.""",0.833333,1.000000,1.000000
3,"""What is the purpose of using a study journal in your scripture study?""","""The purpose of using a study journal in your scripture study is to help you understand, clarify, and remember what you are learning. It also serves as a tool to recall spiritual experiences, gain new insights, and recognize personal growth.""",1.000000,1.000000,1.000000
4,"""What is the importance of organizing and summarizing lesson plans for effective teaching?""","""Organizing and summarizing lesson plans is crucial for effective teaching as it helps in presenting the doctrine clearly and with spiritual power. By structuring the content into a lesson plan, focusing on key points such as doctrine, baptismal interview questions, and commitments, one can deliver the material in a coherent and impactful manner. Additionally, enhancing the lesson plan with scriptures, questions, and examples tailored to the teaching appointments' length can further engage the audience and facilitate better understanding and retention of the teachings.""",0.666667,0.888889,1.000000
5,"""What is the recommended method for highlighting key words when marking scriptures?""","""Circle or underline key words, and then use straight lines to link closely related words.""",1.000000,1.000000,1.000000
6,"""What is the significance of beginning study activities with a prayer?""","""Beginning study activities with a prayer is significant because it sets a foundation of faith and personal agency for the study session. It helps in seeking guidance, inspiration, and spiritual strength to make the study time more rewarding and spiritually uplifting.""",0.833333,1.000000,1.000000
7,"""What is the significance of the power of ordination in the context of missionary work?""","""The significance of the power of ordination in the context of missionary work is that missionaries are authorized to preach the gospel and administer its ordinances. This power allows them to fulfill sacred duties, serve others, and help build up the Church. When used worthily, the power of God is manifest through the priesthood authority given to missionaries.""",0.833333,1.000000,1.000000
8,"""What is the purpose of marking scriptures in relation to applying gospel teachings?""","""The purpose of marking scriptures is to assist in thinking deeply about a passage or doctrine of the gospel, helping individuals focus on applying what they learn and teaching others effectively.""",0.833333,1.000000,1.000000
9,"""What is the significance of preaching the gospel according to President Lorenzo Snow?""","""The significance of preaching the gospel, a

In [19]:
tru.get_leaderboard()

,,Context Relevance,Groundedness,Answer Relevance,latency,total_cost
app_name,app_version,,,,,
LlamaIndex_App,03_Advanced_Retriever,0.815789,0.976608,0.947368,18.85,0.000634
